## Structured Streaming Example

### Configuration Spark

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

conf = SparkConf()

conf.setAppName("Structured Streaming Example") # Name of spark application, useful for logs
conf.set("spark.hadoop.fs.s3a.endpoint","http://minio:9000") # Container and Port from MinIO
conf.set("spark.hadoop.fs.s3a.access.key", "admin") # Login from MinIO
conf.set("spark.hadoop.fs.s3a.secret.key", "@admin123") # Password from MinIO
conf.set("spark.hadoop.fs.s3a.path.style.access", True) # Enforces the use of URLs as the format. Without this, Spark attempts to use the AWS standard (bucket.endpoint), which fails in MinIO
conf.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") # Talk to Hadoop/Spark to use new conector S3A
conf.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") # How to credentials are acess via config(access key + secret)
conf.set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") # active extension from Delta Lake
conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") # Change the standard catalog from spark to Delta 
conf.set("hive.metastore.uris", "thrift://metastore:9083") # Connect to Hive Metastore external

spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()

### Paths for Streaming

In [2]:
input_directory = "s3a://landing/cars/*.json" # The location where Spark will monitor for incoming files

checkpoint_directory = "s3a://landing/cars/checkpoint" # The checkpoint is important because Spark needs to know which file was processed last

### Define Json files schema and creating a function to connect on database via jdbc for streaming process

In [7]:
if __name__ == "__main__":
    
    json_schema = StructType([
        StructField("idcar", IntegerType(), True),
        StructField("name", StringType(), True),
        StructField("price", StringType(), True),
        StructField("situation", StringType(), True)
    ])
    
    data_frame = spark.readStream.schema(json_schema).json(input_directory)
    
    def update_postgres(df, BatchId): # "BatchId" is necessary for foreachBatch function on streaming query
        try:
            df.write.format("jdbc")\
            .option("url", "jdbc:postgresql://postgres_adventureworks:5432/Adventureworks")\
            .option("dbtable", "table_cars")\
            .option("user", "postgres")\
            .option("password", "postgres")\
            .option("driver", "org.postgresql.Driver")\
            .mode("append")\
            .save()
        except Exception as e:
            print(f"Error during batch write: {str(e)}")

### Creating a Stream Query to write on postgreSQL when files coming in the minIO landing

In [8]:
streamingQuery = data_frame.writeStream.foreachBatch(update_postgres)\
    .outputMode("append")\
    .trigger(processingTime = "5 seconds")\
    .option("checkpointLocation", checkpoint_directory)\
    .start()

streamingQuery.awaitTermination()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.5-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/usr/local/spark/python/lib/py4j-0.10.9.5-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/opt/conda/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 